# QLearning

## Explication de l'algorithme

L'objectif du QLearning est d'apprendre la fonction Qualité qui à une paire (état, action) associe le gain moyen obtenu si l'agent effectue l'action dans cet état.

Afin de représenter son approximation de la fonction Qualité, le QLearning fait usage d'un tableau.

Chaque case de ce tableau est associée à une paire (état, action).

À chaque step d'entraînement, on modifie une unique case du tableau de façon à préciser l'estimation de la fonction Qualité.

Le QLearning est "**off-policy**", il n'agit pas de la même façon en entraînement et hors entraînement.

- **Hors entraînement**, il suit la policy "**greedy**" : choisir l'action menant à la plus grande récompense d'épisode (sur la base de ses connaissances).

- **En entraînement**, il suit la policy "**epsilon-greedy**" : avec probabilité epsilon j'agis aléatoirement (explorer), sinon j'applique la policy greedy (exploiter).

epsilon est une valeure que nous pouvons influencer à l'aide des paramètres.

Le QLearning possède des paramètres que l'on peut regrouper en 2 groupes :
- Les paramètres concernant l'actualisation du tableau
- Ceux qui s'occupent du problème d'exploration/exploitation

**Paramètres actualisation du tableau** :
- **alpha** (ou LearningRate) : Désigne à quel point ce qui vient d'être observé à de l'importance par rapport à ce qui est déjà connu.

- **gamma** : Désigne l'importance accordé aux récompenses obtensibles dans le futur

**Par défaut** :
- alpha = 0.1
- gamma = 0.9

**Paramètres exploration/exploitation**

Ces paramètres influencent espsilon (la probabilité d'explorer) :
- **eps_start** : Valeur d'epsilon en début d'entraînement
- **eps_end** : Valeur d'epsilon à l'issu de sa décroissance
- **eps_fraction** : La fraction de l'entraînement au cours de laquelle epsilon décroit de eps_start jusqu'à eps_end.


**Exemple** :

Si nous avons eps_start = 1, eps_end = 0 et eps_fraction = 0.5 et que l'entraînement se fait sur 100 timesteps.

Alors au bout de 100*0.5 = 50 timesteps, epsilon vaudra 0.
Comme la décroissance de epsilon est linéaire alors, au timesteps 25 de l'entraînement la valeur de epsilon sera de 0.5.


**Attention à ne pas définir eps_end à 0, sinon on perd des garanties de convergence**


**Valeurs par défaut**
- eps_start = 0.9
- eps_end = 0.05
- eps_fraction = 0.3

# QLearning : Entraînement

Comment utiliser notre implémentation du QLearning pour entraîner un agent ?

## Utilisation basique

In [ ]:
from QLearning import *

env = creer_instance_environnement()

model, gains, _ = train_q_learning(env, timesteps = 500)

La méthode train_q_learning() permet de débuter un nouvel entraînement (impossible de reprendre un entraînement en cours)

Cette méthode prend en paramètre l'environnement sur lequel entraîner l'agent ainsi qu'un nombre de steps d'entraînement.

Au cours de ces steps d'entraînement, l'agent termine potentiellement de multiples épisodes.
À chaque fin d'épisode le gain obtenu est conservé dans une liste, cette liste est renvoyer (nommée gains dans l'exemple)

**gains** correpond aux gains par épisode pour tous les épisodes effectués pendant l'entraînement

**model** correspond au tableau final obtenu que l'on pourra utiliser en dehors de l'entraînement.

## Gains hors entraînement, réduire le bruit des gains observés

In [ ]:
from QLearning import *

env = creer_instance_environnement()

model, gains, gainExploitation = train_q_learning(env, 
                                                  timesteps = 500, 
                                                  useProdForReward = True, 
                                                  maxTimesteps = 200)

Comme pour le cas d'utilisation classique, on passe l'environnement ainsi qu'un nombre de steps.
Et on obtient le **model** ainsi que les **gains** par épisode.

Or, le QLearning est une méthode off-policy, l'agent ne se comporte pas de la même façon selon s'il s'entraîne ou non.

Le comportement de l'agent pendant l'entraînement est plus aléatoire ce qui peut ammener à du bruit dans les gains.

Afin de mieux mesurer les performances de l'agent au fil de son entraînement, nous pouvons préciser la paramètre **useProdForReward** à True.

Si cela est fait, alors à chaque fin d'épisode pendant l'entraînement, on calcul un nouvel épisode d'évaluation avec le comportement hors entraînement de l'agent et on conserve son gain.

**gainExploitation** correspond à la liste des gains obtenus pendant ces épisodes d'évaluation

Attention à ne pas oublier de préciser **maxTimesteps** qui permet de définir le nombre maximal de steps avant de tronquer les épisodes d'évaluation ce qui permet d'éviter les boucles infinies.

## Paramétrage de l'entraînement

Pour paramétrer QLearning, il suffit de transmettre la valeur des paramètres pendant l'appel à train_q_learning()

In [ ]:
from QLearning import *

env = creer_instance_environnement()

# Exemple de paramétrage 
model, gains, _ = train_q_learning(env, timesteps = 100,
                                   alpha = 0.01,
                                   gamma = 0.99,
                                   eps_start = 1,
                                   eps_end = 0.05,
                                   eps_fraction = 0.7)

# QLearning : Exploitation

## Usage classique

Après avoir entraîné un agent et obtenu son tableau (le modèle), nous pouvons l'utiliser pour observer des épisodes à l'aide de test_q_learning().

In [ ]:
from QLearning import *

env = monEnvironnement()

model = train(....)



gain = test_q_learning(env, model)

La méthode test_q_learning prend en paramètre un environnement ainsi qu'un modèle et déroule un unique épisode.

À chaque step, test_q_learning() utilise la méthode env.render() afin d'effectuer l'affichage.

Puis renvoie le gain obtenu par l'agent au cours de l'épisode.

## Tronquer l'épisode

Il peut arriver que certains comportements dans certains environnements ammènnent à des boucles infinies.

Pour éviter cela, il faut tronquer l'épisode à partir d'un certain nombre de steps à l'aide de maxTimesteps.

In [ ]:
from QLearning import *

env = monEnvironnement()

model = train(....)



gain = test_q_learning(env, model, maxTimesteps = 200)  # Effectue au plus 200 steps

## Ne pas render l'épisode

Si la méthode render() de l'environnement n'est pas implémentée ou si vous ne souhaitez pas effectuer l'affichage, vous pouvez préciser le paramètre **render** à False.

In [ ]:
from QLearning import *

env = monEnvironnement()

model = train(....)


# Effectue au plus 200 steps et n'effectue pas l'affichage
gain = test_q_learning(env, model, maxTimesteps = 200, render = False)

# Quels environnements sont compatibles avec le QLearning ?

Notre implémentation du QLearning est compatible avec les environnements ayant pour attribut :

Un **action_space** du type spaces.**Discrete**(n)

Un **observation_space** du type :
- spaces.**Discrete**(n)
- OU
- spaces.**MultiDiscrete**([n1, n2, n3])